# Глубинное обучение на табличных данных

Начнем конечно с импортов различных, которые понадобятся далее. Также, многое отсюда было реализовано с помощью функций копипаста с семинаров, что мы проходили, помимо этого, семинары дали вдохновление на именно такой пайплайн действий) Стоило упомянуть

In [29]:
import pandas as pd
import numpy as np
import random

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader #чтобы подавать данные батчами

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score #для подсчета качества

import os
import wandb

Объявим класс конфига

In [30]:
class CFG:
  project = "kolesa-cars"
  entity = "armntvs-d3v-student"
  num_epochs = 15
  train_batch_size = 64
  test_batch_size = 256
  lr = 0.001
  seed = 42
  wandb = True

In [31]:
# конфиг в словарь
def class2dict(f):
  return dict((name, getattr(f, name)) for name in dir(f) if not name.startswith('__'))

In [32]:
# вход в W&B (ключ спросит один раз)
wandb.login()

True

Зафиксируем сиды

In [33]:
def seed_everything(seed): # чтобы запуски были стабильными и воспроизводимымми
    random.seed(seed) # фиксируем генератор случайных чисел
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU

In [34]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device # то есть, если есть видеокарта то юзаем cuda, если нет - то юзаем cpu

device(type='cpu')

Загрузим все предобработанные данные 

In [35]:
X_train =pd.read_csv('../data/X_train.csv')
X_test =pd.read_csv('../data/X_test.csv')
y_train =pd.read_csv('../data/y_train.csv')
y_test =pd.read_csv('../data/y_test.csv')

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((4236, 570), (1060, 570), (4236, 1), (1060, 1))

Когда переводил в тензоры выдавало ошибку, потому что нужны флоат, а там видимо от ohe остались bool, переведем это все в флоат и пойдем дальше

In [36]:
X_train = X_train.astype(float)
X_test = X_test.astype(float)
y_train = y_train.astype(float)
y_test = y_test.astype(float)

Переведем все в тензоры

In [37]:
X_train_tensor =torch.FloatTensor(X_train.values)
X_test_tensor =torch.FloatTensor(X_test.values)
y_train_tensor =torch.FloatTensor(y_train.values)
y_test_tensor =torch.FloatTensor(y_test.values)

X_train_tensor.shape, y_train_tensor.shape

(torch.Size([4236, 570]), torch.Size([4236, 1]))

Теперь соединим это обратно в датасеты (train/test), чтобы передавать в модели именно все признаки с таргетом, после чего создадим dataloader чтобы подавать в модель данные батчами/частями

In [38]:
train_ds = TensorDataset(X_train_tensor,y_train_tensor)
test_ds = TensorDataset(X_test_tensor,y_test_tensor)

train_ds

In [39]:
train_loader = DataLoader(train_ds, batch_size= CFG.train_batch_size, shuffle=True) #shuffle тут чтобы между эпохами перемешивались строки и модель не привыкала к порядку
test_loader = DataLoader(test_ds, batch_size= CFG.test_batch_size)


In [40]:
input_size = X_train.shape[1]
input_size 

570

Сейчас построим достаточно базовую модель (вдохновление от 15 семинара). Архитектура простая - 3 полносвязных слоя и все. Далее будем строить несколько более сложных архитектур. Сначала простую чтобы понимать, помогают ли нам новые архитектуры улучшить так скажем 'базовое' качество. 

In [41]:
class Simple(nn.Module): # наследуемся от класса nn.Module
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 128) # первый скрытый слой - 128 нейронов
        self.fc2 = nn.Linear(128, 64) # второй скрытый - 64
        self.fc3 = nn.Linear(64, 1) # третий  выходной- 1 прогноз
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x) 
        return x

Теперь построим более сложную (чуть) архитектуру. Попробуем добавить просто больше скрытых слоев и добавим еще больше нейронов 

Но, при этом, есильно выше риск переобучения, поэтому будем все это сравнивать на качестве теста

In [42]:
class Deep(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        
        self.fc1 = nn.Linear(input_size, 512) #теперь начинаем уменьшение размерности с 512 
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = self.fc5(x)
        return x

И, добавим третью модель, тут мы добавим еще дропаут и батчнорм. Первый будет бороться с переобучением за счет отключения определенного количества нейронов во время обучения. Батчнорм будет просто стабилизировать значения внутри срктытых слоев

In [43]:
class Regularized(nn.Module):
    def __init__(self, input_size):
        super().__init__()


        self.net = nn.Sequential(nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.05), #1 скрытый 
                                 nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),nn.Dropout(0.05),  #2 скрытый
                                 nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))    #3 скрытый и 4 выходной
        #sequential тут потому что во первых мы так работали на семинарах, а во-вторых потому что слои и операции в них просто идут по порядку
        #без сложностей , и следующая функция красивая в 2 строки получается )))

    def forward(self, x):
        return self.net(x)

Мы решили добавить еще одну полнсвязную сеть с residual connection. Его преимущество в том, что он сохраняет вход блока и добавляет к выходу. То есь, это хорошо потому что мы по идее не теряем юзфул информацию когда проходим несколько слоев. В нашем случае, это полезно потому что после ohe у нас стало много признаков, и там есть важные признаки ,которые residual mlp поможет учитывать в каких то сложных комбинациях

Сайты, в том числе откуда и вдохновение на код:
- https://discuss.pytorch.org/t/how-to-use-residual-learning-applied-to-fully-connected-networks/98708/5
- https://stackoverflow.com/questions/60817390/implementing-a-simple-resnet-block-with-pytorch

In [44]:
class ResBlock(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        
    def forward(self, x):
        residual = x  # сохраняем вход блока
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        x = x + residual  # прибавляем старый x к новому x
        x = torch.relu(x)
        
        return x

In [45]:
class ResMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        
        self.fc1 = nn.Linear(input_size, 256)
        self.block1 = ResBlock(256)
        self.block2 = ResBlock(256)
        self.fc2 = nn.Linear(256, 1)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.block1(x)
        x = self.block2(x)
        x = self.fc2(x)
        return x

Далее, переходим к следующей части , пока просто зададим функцию потерь и потом напишем фукнцию обучения

In [46]:
criterion = nn.MSELoss()

In [47]:
from tqdm import tqdm

def train(model, device, train_loader, optimizer, criterion, epoch, WANDB): 
    model.train()
    train_loss = 0
    
    for data, target in tqdm(train_loader):
        data = data.to(device)
        target = target.to(device)
        
        optimizer.zero_grad() #обнуяем градиенты
        
        output = model(data) #прямой проход
        loss = criterion(output, target) #считаем ошибку
        
        loss.backward() #обратынй проход
        optimizer.step() #шаг оптимизатором
        train_loss += loss.item()
    
    train_loss = train_loss / len(train_loader)
    
    tqdm.write('\nTrain set: Average loss: {:.4f}'.format(train_loss)) #как в семе, но чуть поправленная, потому что там для картинок

    if WANDB:
        wandb.log({'epoch': epoch,
                   'train_loss': train_loss})
        
    return train_loss

Теперь функция тестирования, оч похожа на сем, но там картинки

In [48]:
def test(model, device, test_loader, criterion, epoch, WANDB):
    model.eval()  
    
    test_loss = 0
    preds = []
    true = []
    
    # показываем, что обучения нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            
            output = model(data)
            test_loss += criterion(output, target).item()
            
            preds.append(output.cpu())
            true.append(target.cpu())
    
    test_loss = test_loss / len(test_loader)
    
    preds = torch.cat(preds).numpy().ravel() #тут мы клеим прогнозы в одномерный массив нампай 
    true = torch.cat(true).numpy().ravel()
    
    preds = np.expm1(preds) #возвращаем логарифмированные числа в первоначальное 
    true = np.expm1(true)
    
    mae = mean_absolute_error(true, preds)
    rmse = np.sqrt(mean_squared_error(true, preds))
    r2 = r2_score(true, preds)
    
    tqdm.write('Test set: Average loss: {:.4f}, MAE: {:.2f}, RMSE: {:.2f}, R2: {:.4f}'.format(
       test_loss, mae, rmse, r2))
    
    if WANDB:
        wandb.log({'epoch': epoch,
                   'test_loss': test_loss,
                   'test_mae': mae,
                   'test_rmse': rmse,
                   'test_r2': r2})
    
    return test_loss, mae, rmse, r2

Теперь нужна функция запуска наших экспериментов, и после этого можно будет лицезреть качество выстроенных нами моделей))

In [49]:
# основная функция для экспериментов
def main(model, model_name):

    if CFG.wandb:
        wandb.init(project=CFG.project, entity=CFG.entity, name=model_name, reinit=True, config=class2dict(CFG)) #reinit чтобы каждый экспер записывался как новый
        # параметры архитектуры https://docs.wandb.ai/guides/track/config
        wandb.config.update({'model': str(model)})
    seed_everything(CFG.seed)  # фиксируем сиды
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.lr)
    
    train_losses = []
    test_losses = []
    best_mae =10**20
    
    if CFG.wandb:
        wandb.watch(model, log='all') # логируем все (метрики, лоссы, градиенты)


    for epoch in range(1, CFG.num_epochs + 1):
        print('\nEpoch:', epoch)
        train_loss = train(model, device, train_loader, optimizer, criterion, epoch, CFG.wandb)
        test_loss, mae, rmse, r2 = test(model, device, test_loader, criterion, epoch, CFG.wandb)
        train_losses.append(train_loss)
        test_losses.append(test_loss)

        if mae < best_mae:
            best_mae = mae
            best_test_loss = test_loss
            best_rmse = rmse
            best_r2 = r2
    
    print('Training is ended!')
    
    
    if CFG.wandb:
        # сохраняем модель артефактом https://docs.wandb.ai/guides/artifacts
        torch.save(model.state_dict(), f'../data/models/{model_name}.pt')
        artifact = wandb.Artifact(model_name, type='model')
        artifact.add_file(f'../data/models/{model_name}.pt')
        wandb.log_artifact(artifact)
        wandb.finish()

    return model, train_losses, test_losses, best_test_loss, best_mae, best_rmse, best_r2

In [50]:
# данные в W&B для воспроизводимости https://docs.wandb.ai/guides/artifacts
# выполнить один раз
if CFG.wandb:
    wandb.init(project=CFG.project, entity=CFG.entity, name='dataset-tabular', reinit=True)
    art = wandb.Artifact('kolesa-tabular', type='dataset')
    for f in ['X_train.csv', 'X_test.csv', 'y_train.csv', 'y_test.csv']:
        art.add_file(f'../data/{f}')
    wandb.log_artifact(art)
    wandb.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


IOStream.flush timed out
IOStream.flush timed out
wandb: WARNING Artifact "kolesa-tabular" already exists with the same content. No new version will be created.


In [ ]:
seed_everything(CFG.seed)
simple_model, simple_train_losses, simple_test_losses, simple_test_loss, simple_mae, simple_rmse, simple_r2 = main(Simple(input_size),'Simple')


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 1536.56it/s]



Train set: Average loss: 162.5042
Test set: Average loss: 4.0265, MAE: 39311080.00, RMSE: 101898886.22, R2: -99.8154

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 1235.18it/s]



Train set: Average loss: 3.2121
Test set: Average loss: 1.8561, MAE: 13277700.00, RMSE: 24782195.99, R2: -4.9630

Epoch: 3


...


Train set: Average loss: 0.1055
Test set: Average loss: 0.2991, MAE: 2940792.75, RMSE: 6653108.86, R2: 0.5702

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 1708.22it/s]



Train set: Average loss: 0.0897
Test set: Average loss: 0.2809, MAE: 2790402.75, RMSE: 6643852.15, R2: 0.5714
Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇██
test_loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁
test_mae,█▃▂▂▂▂▁▁▁▁▁▁▁▁▁
test_r2,▁██████████████
test_rmse,█▂▂▂▂▂▁▁▁▁▁▁▁▁▁
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,15
test_loss,0.28093
test_mae,2790402.75
test_r2,0.57142
test_rmse,6643852.15142


In [ ]:
seed_everything(CFG.seed)
deep_model,deep_train_losses, deep_test_losses, deep_test_loss, deep_mae,deep_rmse, deep_r2 = main(Deep(input_size), 'Deep')


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 771.92it/s]



Train set: Average loss: 71.0038
Test set: Average loss: 1.7153, MAE: 21532352.00, RMSE: 52541648.79, R2: -25.8037

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 896.81it/s]



Train set: Average loss: 0.9890
Test set: Average loss: 0.7606, MAE: 7256099.00, RMSE: 16550734.83, R2: -1.6596

Epoch: 3


...

100%|██████████| 67/67 [00:00<00:00, 839.86it/s]



Train set: Average loss: 0.0488
Test set: Average loss: 0.2202, MAE: 2554039.00, RMSE: 6214484.15, R2: 0.6250
Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: WARNING Artifact "Deep" already exists with the same content. No new version will be created.


epoch,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇██
test_loss,█▄▂▂▂▁▁▁▁▁▁▁▁▁▁
test_mae,█▃▂▂▂▂▁▁▁▁▁▁▁▁▁
test_r2,▁▇█████████████
test_rmse,█▃▂▂▂▂▁▁▁▁▁▁▁▁▁
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,15
test_loss,0.22015
test_mae,2554039.0
test_r2,0.62503
test_rmse,6214484.14987


In [ ]:
seed_everything(CFG.seed)
regularized_model,regularized_train_losses, regularized_test_losses, regularized_test_loss, regularized_mae,regularized_rmse, regularized_r2 = main(
    Regularized(input_size),'Regularized')


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 603.27it/s]



Train set: Average loss: 103.7855
Test set: Average loss: 2.5298, MAE: 8499811.00, RMSE: 12250580.68, R2: -0.4571

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 626.61it/s]



Train set: Average loss: 1.6608
Test set: Average loss: 0.6946, MAE: 5825539.50, RMSE: 9206315.61, R2: 0.1771

Epoch: 3


...

100%|██████████| 67/67 [00:00<00:00, 751.52it/s]



Train set: Average loss: 0.5216
Test set: Average loss: 0.2259, MAE: 3761167.00, RMSE: 6604677.75, R2: 0.5765
Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: WARNING Artifact "Regularized" already exists with the same content. No new version will be created.


epoch,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇██
test_loss,█▂▂▂▂▁▁▂▁▁▁▁▁▁▁
test_mae,█▄▄▂▃▂▂▃▃▂▂▁▂▂▁
test_r2,▅▇▇▆▇▆█▇▁█▇██▇█
test_rmse,▅▃▃▃▂▃▂▂█▁▂▁▂▃▁
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,15
test_loss,0.22587
test_mae,3761167.0
test_r2,0.57646
test_rmse,6604677.75391


In [ ]:
seed_everything(CFG.seed)
res_model, res_train_losses, res_test_losses, res_test_loss, res_mae, res_rmse, res_r2 = main(ResMLP(input_size), 'ResMLP')


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 599.46it/s]



Train set: Average loss: 43.8960
Test set: Average loss: 0.8779, MAE: 12304962.00, RMSE: 41029536.93, R2: -15.3449

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 484.77it/s]



Train set: Average loss: 0.5527
Test set: Average loss: 0.4417, MAE: 5516935.50, RMSE: 13631449.23, R2: -0.8041

Epoch: 3


...

100%|██████████| 67/67 [00:00<00:00, 540.93it/s]



Train set: Average loss: 0.0366
Test set: Average loss: 0.1667, MAE: 2407291.50, RMSE: 5291116.50, R2: 0.7282
Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: WARNING Artifact "ResMLP" already exists with the same content. No new version will be created.


epoch,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇██
test_loss,█▄▃▂▁▁▁▁▁▁▁▁▁▁▁
test_mae,█▃▂▂▂▁▁▁▁▁▁▁▂▁▁
test_r2,▁▇█████████████
test_rmse,█▃▂▂▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,15
test_loss,0.16671
test_mae,2407291.5
test_r2,0.72818
test_rmse,5291116.50285


In [55]:
res = pd.DataFrame([{'model': 'Simple', 'test_loss': simple_test_loss, 'MAE': simple_mae, 'RMSE': simple_rmse, 'R2': simple_r2},
                        {'model': 'Deep', 'test_loss': deep_test_loss, 'MAE': deep_mae, 'RMSE': deep_rmse, 'R2': deep_r2},
                        {'model': 'Regularized', 'test_loss': regularized_test_loss, 'MAE': regularized_mae, 'RMSE': regularized_rmse, 'R2': regularized_r2},
                        {'model': 'ResMLP', 'test_loss': res_test_loss, 'MAE': res_mae, 'RMSE': res_rmse, 'R2': res_r2}])

res

,model,test_loss,MAE,RMSE,R2
0,Simple,0.280934,2790402.75,6.643852e+06,0.571424
1,Deep,0.221377,2392471.50,4.542574e+06,0.799649
2,Regularized,0.225874,3761167.00,6.604678e+06,0.576463
3,ResMLP,0.166707,2407291.50,5.291117e+06,0.728179


In [56]:
res.sort_values('MAE')

,model,test_loss,MAE,RMSE,R2
1,Deep,0.221377,2392471.50,4.542574e+06,0.799649
3,ResMLP,0.166707,2407291.50,5.291117e+06,0.728179
0,Simple,0.280934,2790402.75,6.643852e+06,0.571424
2,Regularized,0.225874,3761167.00,6.604678e+06,0.576463


### Сравнение моделей и выводы

По ходу работы мы обучили 4 модели, от простой базовой полносвязной сети до полносвязной с residual блоками. Основная метрика - считаем MAE, средняя ошибка прогноза цены в тенге. 

По результатам: лучшей стала модель ResMLP с показателями MAE - примерно 2,29 млн тенге, R2 - примерно 0,82, то есть модель лучше остальных отличает цены и в среднем меньше ошибается. Базовая модель(Simple) и модель с бОльшим количеством скрытых слоев(Deep) также дают хороший результат. Худший результат показала модель Regularized, возможно dropout для нашего датасета мешала модели запоминать какие-то важные зависимости в данных или еще что-то, но любые манипуляции с весами не помогали. 